### Reed API Manual Data Collection

In [10]:
#import libraries
import os
import math
import time
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth

In [11]:
# Reed API Configuration

API_KEY = " 5ca6123b-b48b-4d3f-9f75-2b34e8aa2217"

BASE_URL = "https://www.reed.co.uk/api/1.0/search"

RESULTS_PER_PAGE = 100

In [3]:
os.makedirs("data/raw/reed", exist_ok=True)

In [12]:
# SECTOR DEFINITIONS
# Multiple keywords improve coverage

SECTORS = {

    "it": [
        "software engineer",
        "software developer",
        "programmer",
        "web developer",
        "IT support",
        "network engineer",
        "systems administrator"
    ],

    "data": [
        "data scientist",
        "data analyst",
        "machine learning",
        "AI engineer",
        "business intelligence"
    ],

    "engineering": [
        "engineering",
        "mechanical engineer",
        "civil engineer",
        "electrical engineer"
    ],

    "finance": [
        "finance",
        "accountant",
        "financial analyst",
        "banking"
    ],

    "healthcare": [
        "healthcare",
        "nurse",
        "doctor",
        "care assistant",
        "pharmacist"
    ],

    "education": [
        "teacher",
        "lecturer",
        "education",
        "teaching assistant"
    ],

    "sales": [
        "sales",
        "sales executive",
        "sales assistant"
    ],

    "marketing": [
        "marketing",
        "digital marketing",
        "SEO",
        "social media"
    ],

    "retail": [
        "retail",
        "store manager",
        "cashier"
    ],

    "construction": [
        "construction",
        "builder",
        "site manager"
    ],

    "logistics": [
        "logistics",
        "warehouse",
        "supply chain",
        "delivery driver"
    ],

    "hospitality": [
        "hospitality",
        "chef",
        "waiter",
        "hotel"
    ],

    "hr": [
        "human resources",
        "HR",
        "recruitment"
    ],

    "legal": [
        "lawyer",
        "solicitor",
        "legal assistant"
    ],

    "manufacturing": [
        "manufacturing",
        "production",
        "factory"
    ]

}

In [13]:
# DOWNLOAD FUNCTION

def download_keyword(keyword):

    print(f"\nSearching: {keyword}")

    response = requests.get(
        BASE_URL,
        auth=HTTPBasicAuth(API_KEY, ""),
        params={
            "keywords": keyword,
            "resultsToTake": RESULTS_PER_PAGE,
            "resultsToSkip": 0
        }
    )

    if response.status_code != 200:
        print("Cannot retrieve first page.")
        return []

    data = response.json()

    total_jobs = data["totalResults"]

    jobs = data["results"]

    print("Total Jobs:", total_jobs)

    total_pages = math.ceil(total_jobs / RESULTS_PER_PAGE)

    # Reed pagination limit
    max_pages = min(total_pages, 100)

    print("Pages to collect:", max_pages)

    all_jobs = jobs

    for page in range(1, max_pages):

        skip = page * RESULTS_PER_PAGE

        retries = 3

        while retries > 0:

            response = requests.get(
                BASE_URL,
                auth=HTTPBasicAuth(API_KEY, ""),
                params={
                    "keywords": keyword,
                    "resultsToTake": RESULTS_PER_PAGE,
                    "resultsToSkip": skip
                }
            )

            if response.status_code == 200:

                page_jobs = response.json()["results"]

                all_jobs.extend(page_jobs)

                print(
                    f"Page {page+1}/{max_pages} | "
                    f"Collected {len(all_jobs)} jobs"
                )

                break

            else:

                retries -= 1

                print(
                    f"Retry... Page {page+1}"
                )

                time.sleep(2)

        time.sleep(1)

    return all_jobs

In [7]:
# Define the output folder
OUTPUT_FOLDER = "data/raw/reed" # You can change this to your desired folder path

# Create the output directory if it doesn't exist
os.makedirs("data/raw/reed", exist_ok=True)  

# MAIN LOOP
for sector, keywords in SECTORS.items():

    print("\n")
    print("=" * 70)
    print("SECTOR:", sector.upper())
    print("=" * 70)

    sector_jobs = []

    for keyword in keywords:

        jobs = download_keyword(keyword)

        sector_jobs.extend(jobs)

    df = pd.DataFrame(sector_jobs)

    if len(df) == 0:

        print("No jobs found.")

        continue

    # Remove duplicates

    if "jobId" in df.columns:

        before = len(df)

        df.drop_duplicates(
            subset="jobId",
            inplace=True
        )

        after = len(df)

        print(
            f"Duplicates removed: {before-after}"
        )

    output_file = os.path.join(
        OUTPUT_FOLDER,  # Now this variable is defined
        f"{sector}_jobs.csv"
    )

    df.to_csv(
        output_file,
        index=False
    )

    print(f"Saved: {output_file}")
    print(f"Final Jobs: {len(df)}")

print("\n")
print("=" * 70)
print("ALL SECTORS COMPLETED")
print("=" * 70)



SECTOR: IT

Searching: software engineer
Total Jobs: 2687
Pages to collect: 27
Page 2/27 | Collected 200 jobs
Page 3/27 | Collected 300 jobs
Page 4/27 | Collected 400 jobs
Page 5/27 | Collected 500 jobs
Page 6/27 | Collected 600 jobs
Page 7/27 | Collected 700 jobs
Page 8/27 | Collected 800 jobs
Page 9/27 | Collected 900 jobs
Page 10/27 | Collected 1000 jobs
Page 11/27 | Collected 1100 jobs
Page 12/27 | Collected 1200 jobs
Page 13/27 | Collected 1300 jobs
Page 14/27 | Collected 1400 jobs
Page 15/27 | Collected 1500 jobs
Page 16/27 | Collected 1600 jobs
Page 17/27 | Collected 1700 jobs
Page 18/27 | Collected 1800 jobs
Page 19/27 | Collected 1900 jobs
Page 20/27 | Collected 2000 jobs
Page 21/27 | Collected 2100 jobs
Page 22/27 | Collected 2200 jobs
Page 23/27 | Collected 2300 jobs
Page 24/27 | Collected 2400 jobs
Page 25/27 | Collected 2500 jobs
Page 26/27 | Collected 2600 jobs
Page 27/27 | Collected 2687 jobs

Searching: software developer
Total Jobs: 1636
Pages to collect: 17
Page 2/1